In [31]:
import sys

sys.path.append("..")
import numpy as np
import os
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import plotly.express as px
from classes.sample.raw_data import LMSFile
from classes.signal.signal_classes import SignalSpectrum
from classes.sample.sample_classes import SinglePSDSample, SamplePSD
import json
import scipy.signal as si
from typing import Tuple
from tqdm import tqdm

In [3]:
path = Path(
    os.getenv("RAW_DATA_PATH", "data/raw")
)

In [4]:
folders = [
    folder
    for folder in os.listdir(path)
    if folder.startswith("Teste") and not folder.endswith(".npy")
]

In [7]:
folder = folders[10]
files = os.listdir(path / folder)
prefixes = np.unique([f[:-8] for f in files if f.endswith(".txt")])

In [11]:
prefix = prefixes[10]
lms_path = path / folder / f"{prefix}_LMS.txt"
lms_file = LMSFile.from_txt(lms_path)

In [13]:
t = np.arange(0, len(lms_file.c4_z) * lms_file.dt, lms_file.dt)

In [33]:
signals = pd.DataFrame({
    "c4_x": lms_file.c4_x,
    "c4_y": lms_file.c4_y,
    "c4_z": lms_file.c4_z,
    "c3_x": lms_file.c3_x,
    "c3_y": lms_file.c3_y,
    "c3_z": lms_file.c3_z,
    "c2_x": lms_file.c2_x,
    "c2_y": lms_file.c2_y,
    "c2_z": lms_file.c2_z,
    "c1_x": lms_file.c1_x,
    "c1_y": lms_file.c1_y,
    "c1_z": lms_file.c1_z,
    "t": t
})
signals = signals.set_index("t")


In [259]:
win_size = 1024
window = "blackman"
spectrums = signals.apply(lambda x: si.welch(x, fs=1/lms_file.dt, nperseg=win_size, window=window)[1])
spectrums["f"] = si.welch(lms_file.c4_x, fs=1/lms_file.dt, nperseg=win_size, window=window)[0]
spectrums = spectrums.set_index("f")

# spectrums = signals.apply(lambda x: np.abs(np.fft.rfft(x)))
# spectrums["f"] =np.fft.rfftfreq(len(lms_file.c4_x), lms_file.dt)
# spectrums = spectrums.set_index("f")

In [260]:
comb_spectrums = pd.DataFrame({
    "c4": np.sqrt(spectrums["c4_y"]**2 + spectrums["c4_z"]**2),
    "c3": np.sqrt(spectrums["c3_y"]**2 + spectrums["c3_z"]**2),
    "c2": np.sqrt(spectrums["c2_y"]**2 + spectrums["c2_z"]**2),
    "c1": np.sqrt(spectrums["c1_y"]**2 + spectrums["c1_z"]**2),
    "f": spectrums.index
})
comb_spectrums.set_index("f", inplace=True)


In [241]:
cross_spectrums = pd.DataFrame({
    "c4_y_c4_z": si.csd(signals["c4_y"], signals["c4_z"], fs=1/lms_file.dt, nperseg=win_size, window="hamming")[1],
    "c3_y_c3_z": si.csd(signals["c3_y"], signals["c3_z"], fs=1/lms_file.dt, nperseg=win_size, window="hamming")[1],
    "c2_y_c2_z": si.csd(signals["c2_y"], signals["c2_z"], fs=1/lms_file.dt, nperseg=win_size, window="hamming")[1],
    "c1_y_c1_z": si.csd(signals["c1_y"], signals["c1_z"], fs=1/lms_file.dt, nperseg=win_size, window="hamming")[1],
})
cross_spectrums["f"] = si.welch(lms_file.c4_x, fs=1/lms_file.dt, nperseg=win_size, window="hamming")[0]
cross_spectrums = cross_spectrums.set_index("f")

In [233]:
abs_cross_spectrums = cross_spectrums.apply(lambda x: np.abs(x))
phase_cross_spectrums = cross_spectrums.apply(lambda x: np.unwrap(np.angle(x))* 180 / np.pi)


In [234]:
px.line(signals, y=["c4_x", "c4_y", "c4_z"])


In [261]:
px.line(20*np.log10(spectrums))

In [262]:
px.line(20*np.log10(comb_spectrums))

In [216]:
px.line(phase_cross_spectrums, y=["c4_y_c4_z", "c3_y_c3_z", "c2_y_c2_z", "c1_y_c1_z"])

In [190]:
# b, a = si.butter(N=4, Wn=[70, 75], btype='band', fs=1/lms_file.dt)
notch = si.iirnotch(w0=15, Q=1, fs=1/lms_file.dt)
filt_signals = signals.apply(lambda x: si.filtfilt(*notch, x))

In [193]:
px.line(filt_signals, y=["c4_x", "c4_y", "c4_z"])

In [180]:
px.scatter(filt_signals, x="c4_y", y="c4_z")
